# 18 — Prompt Management

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Why Prompt Management Matters

In [3]:
challenges = {
    "Version drift":  "Prompt changed in code -> unexpected behavior in prod",
    "Duplication":    "Same prompt copy-pasted in 10 places -> hard to update",
    "No audit trail": "Who changed what and why? Unknown.",
    "Testing gap":    "Prompts deployed without regression testing",
    "Collaboration":  "PMs, engineers, and data scientists editing same prompts",
}
print("Prompt management challenges:")
for k, v in challenges.items():
    print(f"  {k:20s}: {v}")


Prompt management challenges:
  Version drift       : Prompt changed in code -> unexpected behavior in prod
  Duplication         : Same prompt copy-pasted in 10 places -> hard to update
  No audit trail      : Who changed what and why? Unknown.
  Testing gap         : Prompts deployed without regression testing
  Collaboration       : PMs, engineers, and data scientists editing same prompts


## Prompt as Code: Template Pattern

In [4]:
from string import Template
import json

PROMPT_REGISTRY = {
    "summarize_v1": {
        "version": "1.0",
        "template": "Summarize this text in ${num_sentences} sentences:\n\n${text}",
        "defaults": {"num_sentences": 2},
        "model": "llama-3.1-8b-instant",
        "temperature": 0,
    },
    "classify_v1": {
        "version": "1.0",
        "template": "Classify as ${categories}. Text: '${text}'. Return one word.",
        "defaults": {"categories": "POSITIVE, NEGATIVE, NEUTRAL"},
        "model": "llama-3.1-8b-instant",
        "temperature": 0,
    }
}

def run_prompt(name: str, **kwargs) -> str:
    entry = PROMPT_REGISTRY[name]
    params = {**entry.get("defaults", {}), **kwargs}
    prompt = Template(entry["template"]).substitute(params)
    return chat([{"role":"user","content":prompt}],
                temperature=entry["temperature"], max_tokens=200)

print(run_prompt("summarize_v1",
    text="LLMs are neural networks trained on vast text data. They predict the next token.",
    num_sentences=1
).strip())
print(run_prompt("classify_v1", text="This product exceeded all my expectations!").strip())


Large Language Models (LLMs) are neural networks that have been trained on extensive text data and use this training to predict the next word or token in a sequence.


POSITIVE


## Prompt Catalog

In [5]:
class PromptCatalog:
    def __init__(self):
        self._catalog = {}

    def register(self, name, template, description="", tags=None):
        self._catalog[name] = {
            "template": template,
            "description": description,
            "tags": tags or [],
        }

    def get(self, name, **kwargs):
        return self._catalog[name]["template"].format(**kwargs)

    def search(self, tag):
        return [k for k, v in self._catalog.items() if tag in v["tags"]]

catalog = PromptCatalog()
catalog.register(
    "entity_extract",
    "Extract all {entity_type} from the text. Return as JSON list.\n\nText: {text}",
    description="Named entity extraction",
    tags=["extraction", "nlp"]
)
catalog.register(
    "tone_rewrite",
    "Rewrite the following text in a {tone} tone:\n\n{text}",
    description="Rewrite text in a different tone",
    tags=["writing", "style"]
)

prompt = catalog.get("entity_extract",
    entity_type="company names",
    text="Apple and Microsoft are competing with Google in the AI space.")
result = chat([{"role":"user","content":prompt}], temperature=0, max_tokens=80)
print(result.strip())
print("\nNLP prompts:", catalog.search("nlp"))


{
  "companies": [
    "Apple",
    "Microsoft",
    "Google"
  ]
}

NLP prompts: ['entity_extract']


## Environment-Based Prompt Switching

In [6]:
import os

PROMPTS = {
    "summarize": {
        "dev":  "Quick summary (dev mode): {text}",
        "prod": "You are an expert summarizer. Provide a concise, accurate summary in 2-3 sentences. Maintain factual accuracy.\n\nText: {text}",
    }
}

ENV = os.getenv("APP_ENV", "dev")

def get_prompt(name, env=ENV, **kwargs):
    return PROMPTS[name][env].format(**kwargs)

sample = "Deep learning is a subset of machine learning using multi-layered neural networks."

for env in ["dev", "prod"]:
    p = get_prompt("summarize", env=env, text=sample)
    r = chat([{"role":"user","content":p}], temperature=0, max_tokens=80)
    print(f"[{env.upper()}]: {r.strip()}")


[DEV]: **Deep Learning Summary**

**Dev Mode:**

Deep learning is a subset of machine learning that utilizes complex, multi-layered neural networks to analyze and learn from data. This approach enables the development of sophisticated models capable of recognizing patterns, making predictions, and classifying data with high accuracy.

**Key Characteristics:**

1. **Multi-layered neural networks**: Composed of multiple layers of interconnected nodes
[PROD]: Deep learning is a subset of machine learning that utilizes complex, multi-layered neural networks to analyze and learn from data. This approach enables computers to automatically learn and improve their performance on tasks such as image and speech recognition. By mimicking the structure of the human brain, deep learning models can process and interpret vast amounts of data.
